In [ ]:
from dataclasses import dataclass
from typing import TypedDict, Annotated, Literal

from dotenv import  load_dotenv
from langchain_core.messages import SystemMessage,HumanMessage, AIMessage, ToolMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph,START,END
from langgraph.runtime import Runtime
from loguru import logger
from langgraph.graph.message import MessagesState

load_dotenv(override=True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

@dataclass
class UserContext:
    username:str
    membership_level:str

class OverAllState(MessagesState):
    user_input:str
    output:str

def llm_node(state:OverAllState,runtime:Runtime[UserContext])->OverAllState:

    runtime_context=runtime.context
    if runtime_context:
        level=runtime_context.membership_level
        username=runtime_context.username
        logger.info(f"当前用户：{username}，会员等级：{level}")
        if level == "VIP":
            system_prompt = f"你是高级客户助理,当前VIP用户是{username},请使用尊称'您',语气热情周到,回复末尾加上'VIP🏅服务'"
        else:
            system_prompt = f"你是普通客户助理,当前用户是{username},请友好简洁回复问题"
    else:
        system_prompt = f"你是普通客户助理,请友好简洁回复问题"

    user_input=state["user_input"]
    messages=state.get("messages", [])
    human_message = HumanMessage(content=user_input)
    response = model.invoke([SystemMessage(content=system_prompt)]+messages+[human_message])
    return {
        "messages": [human_message, response],
        "output": response.content
    }

#4. 构建图
builder = StateGraph(state_schema=OverAllState,context_schema=UserContext)

builder.add_node("llm_node",llm_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node",END)

graph = builder.compile()

# ==========第一次调用: 传入VIP上下文用户====================
res = graph.invoke(
    {"user_input":"你好,帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice",membership_level="VIP")
)
print(res)


In [ ]:
# ==========第二次调用: 传入普通上下文用户====================
res1 = graph.invoke(
    {"user_input":"你好,帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice",membership_level="普通用户")
)
print(res1)